# BMM-SCI No.46 — Anomaly Detection Pipeline
**Models**: DAGMM · Isolation Forest · LOF · LSTM Autoencoder  
**CV**: Stratified 5-Fold (by `timing`)  
**Labels**: Real = 0, Injected Anomaly = 1  
**Threshold**: μ + 2σ  
**Optimization**: Bayesian Optimization  
**Analysis**: SHAP dependency plots + contrastive check


## Step 1 — Install Dependencies

In [ ]:
!pip install bayesian-optimization shap -q
print('All packages ready.')

## Step 2 — Upload Dataset & Set Paths
> Drag-and-drop `BMM-SCI. No.46 (Dataset).csv` into the `/content/` folder in the Colab Files panel (left sidebar), then run this cell.

In [ ]:
import os

DATASET_PATH = '/content/BMM-SCI. No.46 (Dataset).csv'
OUTPUT_DIR   = '/content/Reports/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(DATASET_PATH):
    print('Dataset not found — opening upload dialog...')
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded:
        DATASET_PATH = f'/content/{fn}'

print('Dataset :', DATASET_PATH)
print('Exists  :', os.path.exists(DATASET_PATH))
print('Output  :', OUTPUT_DIR)


## Step 3 — Load & Inspect Dataset

In [ ]:
import pandas as pd
import numpy as np

print('Loading...')
df = pd.read_csv(DATASET_PATH)
print('Shape   :', df.shape)
print('Columns :', list(df.columns))
print('Nulls   :\n', df.isnull().sum()[df.isnull().sum() > 0])
print('timing  :\n', df['timing'].value_counts())
print('weekday :\n', df['weekday'].value_counts())
df.head(3)


## Step 4 — Preprocess & Stratified 5-Fold Split

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer

le_timing  = LabelEncoder()
le_weekday = LabelEncoder()
df['timing_enc']  = le_timing.fit_transform(df['timing'])
df['weekday_enc'] = le_weekday.fit_transform(df['weekday'])
print('timing classes :', le_timing.classes_)
print('weekday classes:', le_weekday.classes_)

DROP_COLS    = ['Unnamed: 0', 'time', 'timing', 'weekday']
feature_cols = [c for c in df.columns if c not in DROP_COLS]
X_raw = df[feature_cols].values
strat = df['timing_enc'].values

# Median impute NaN values
global_imp = SimpleImputer(strategy='median')
X = global_imp.fit_transform(X_raw)
print(f'NaN before: {np.isnan(X_raw).sum()}  |  NaN after: {np.isnan(X).sum()}')
print('Feature matrix shape:', X.shape)

skf   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = []
for i, (tr_idx, te_idx) in enumerate(skf.split(X, strat)):
    folds.append({'X_train': X[tr_idx], 'X_test': X[te_idx]})
    td = dict(zip(*np.unique(strat[te_idx], return_counts=True)))
    print(f'Fold {i+1}: train={len(tr_idx)}, test={len(te_idx)}')
print('Done.')


## Step 5 — Anomaly Injection (Test Set Only)
Anomaly magnitude: **1.5–2.5σ** → realistic difficulty, pre-opt F1 ~0.70–0.85

In [ ]:
def inject_anomalies(X_test, ratio=0.1, seed=42):
    rng   = np.random.RandomState(seed)
    n     = len(X_test)
    n_inj = max(1, int(n * ratio))
    Xi    = X_test.copy()
    y     = np.zeros(n, dtype=int)
    idx   = rng.choice(n, n_inj, replace=False)
    mu    = X_test.mean(0)
    sg    = X_test.std(0) + 1e-8
    for j in idx:
        d     = rng.choice([-1, 1], X_test.shape[1])
        Xi[j] = mu + d * sg * rng.uniform(1.5, 2.5, X_test.shape[1])
        y[j]  = 1
    return Xi, y

injected_folds = []
for i, fold in enumerate(folds):
    Xi, yi = inject_anomalies(fold['X_test'], ratio=0.1, seed=42+i)
    injected_folds.append({'X_test_injected': Xi, 'y_true': yi})
    df_inj = pd.DataFrame(Xi, columns=feature_cols)
    df_inj['anomaly_label'] = yi
    df_inj.to_csv(f'{OUTPUT_DIR}Report_Fold{i+1}_Injected_Test_Dataset.csv', index=False)
    print(f'Fold {i+1}: {yi.sum()} anomalies / {len(yi)} samples — saved.')
print('Injection complete.')


## Step 6A — Isolation Forest (CPU, n_jobs=-1)

In [ ]:
from sklearn.ensemble import IsolationForest

IF_results = []
for i, (fold, inj) in enumerate(zip(folds, injected_folds)):
    print(f'IF | Fold {i+1}')
    m = IsolationForest(n_estimators=100, contamination=0.1, n_jobs=-1, random_state=42)
    m.fit(fold['X_train'])
    s = -m.score_samples(inj['X_test_injected'])
    print(f'  score min={s.min():.4f} max={s.max():.4f}')
    IF_results.append({'scores': s, 'y_true': inj['y_true']})
    pd.DataFrame({'anomaly_score': s, 'y_true': inj['y_true']}).to_csv(
        f'{OUTPUT_DIR}Report_Fold{i+1}_IsolationForest_Scores_and_Classes.csv', index=False)
    print('  saved.')
print('IF done.')


## Step 6B — LOF (Subsampled 10K for Speed)

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

N_REF = 10000
LOF_results = []
for i, (fold, inj) in enumerate(zip(folds, injected_folds)):
    print(f'LOF | Fold {i+1}')
    X_tr = fold['X_train']
    if len(X_tr) > N_REF:
        idx = np.random.RandomState(42+i).choice(len(X_tr), N_REF, replace=False)
        X_tr_fit = X_tr[idx]
    else:
        X_tr_fit = X_tr
    m = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.1,
                           algorithm='auto', leaf_size=40, n_jobs=-1)
    m.fit(X_tr_fit)
    s = -m.score_samples(inj['X_test_injected'])
    print(f'  score min={s.min():.4f} max={s.max():.4f}')
    LOF_results.append({'scores': s, 'y_true': inj['y_true']})
    pd.DataFrame({'anomaly_score': s, 'y_true': inj['y_true']}).to_csv(
        f'{OUTPUT_DIR}Report_Fold{i+1}_LOF_Scores_and_Classes.csv', index=False)
    print('  saved.')
print('LOF done.')


## Step 6C — DAGMM (CPU, Fast)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class Encoder(nn.Module):
    def __init__(self, d, z=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,60),nn.Tanh(),nn.Linear(60,30),nn.Tanh(),nn.Linear(30,z))
    def forward(self, x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self, z, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z,30),nn.Tanh(),nn.Linear(30,60),nn.Tanh(),nn.Linear(60,d))
    def forward(self, x): return self.net(x)

class EstNet(nn.Module):
    def __init__(self, d, k=4):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,10),nn.Tanh(),nn.Dropout(0.5),nn.Linear(10,k),nn.Softmax(dim=1))
    def forward(self, x): return self.net(x)

def train_dagmm(Xtr, d, z=2, k=4, epochs=15, bs=1024, lr=1e-3):
    sc  = StandardScaler()
    Xs  = sc.fit_transform(Xtr).astype(np.float32)
    enc = Encoder(d, z)
    dec = Decoder(z, d)
    est = EstNet(z+2, k)
    opt = optim.Adam(list(enc.parameters())+list(dec.parameters())+list(est.parameters()), lr=lr)
    ldr = DataLoader(TensorDataset(torch.from_numpy(Xs)), batch_size=bs, shuffle=True)
    enc.train(); dec.train(); est.train()
    for ep in range(epochs):
        tot = 0
        for (b,) in ldr:
            z_ = enc(b); xh = dec(z_)
            rec = torch.mean((b-xh)**2, dim=1, keepdim=True)
            rel = torch.norm(z_, dim=1, keepdim=True) / (torch.norm(z_, dim=1, keepdim=True)+1e-8)
            g   = est(torch.cat([z_, rec, rel], dim=1))
            loss = torch.mean(rec) + 0.1*torch.mean(g*torch.log(g+1e-8))
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        if (ep+1) % 5 == 0 or (ep+1) == epochs:
            print(f'  ep {ep+1:02d}/{epochs}  loss={tot/len(ldr):.4f}')
    return enc, dec, sc

def score_dagmm(Xte, enc, dec, sc, bs=4096):
    enc.eval(); dec.eval()
    Xs = sc.transform(Xte).astype(np.float32)
    out = []
    with torch.no_grad():
        for i in range(0, len(Xs), bs):
            b  = torch.from_numpy(Xs[i:i+bs])
            z_ = enc(b); xh = dec(z_)
            out.append(torch.mean((b-xh)**2, dim=1).numpy())
    return np.concatenate(out)

DAGMM_results = []
for i, (fold, inj) in enumerate(zip(folds, injected_folds)):
    print(f'DAGMM | Fold {i+1}')
    d = fold['X_train'].shape[1]
    enc, dec, sc = train_dagmm(fold['X_train'], d)
    s = score_dagmm(inj['X_test_injected'], enc, dec, sc)
    print(f'  score min={s.min():.4f} max={s.max():.4f}')
    DAGMM_results.append({'scores': s, 'y_true': inj['y_true']})
    pd.DataFrame({'anomaly_score': s, 'y_true': inj['y_true']}).to_csv(
        f'{OUTPUT_DIR}Report_Fold{i+1}_DAGMM_Scores_and_Classes.csv', index=False)
    print('  saved.')
print('DAGMM done.')


## Step 6D — LSTM Autoencoder (CPU, Fast)

In [ ]:
class LSTMAuto(nn.Module):
    def __init__(self, d, h=64, nl=1):
        super().__init__()
        self.enc_rnn = nn.LSTM(d, h, num_layers=nl, batch_first=True)
        self.dec_rnn = nn.LSTM(h, d, num_layers=nl, batch_first=True)
    def forward(self, x):
        _, (hh, _) = self.enc_rnn(x)
        di = hh[-1].unsqueeze(1).repeat(1, x.size(1), 1)
        out, _ = self.dec_rnn(di)
        return out

def train_lstm(Xtr, d, h=64, nl=1, epochs=15, bs=1024, lr=2e-3):
    sc = StandardScaler()
    Xs = torch.from_numpy(sc.fit_transform(Xtr).astype(np.float32)).unsqueeze(1)
    m  = LSTMAuto(d, h, nl)
    opt  = optim.Adam(m.parameters(), lr=lr)
    crit = nn.MSELoss()
    ldr  = DataLoader(TensorDataset(Xs), batch_size=bs, shuffle=True)
    m.train()
    for ep in range(epochs):
        tot = 0
        for (b,) in ldr:
            out = m(b); loss = crit(out, b)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        if (ep+1) % 5 == 0 or (ep+1) == epochs:
            print(f'  ep {ep+1:02d}/{epochs}  loss={tot/len(ldr):.4f}')
    return m, sc

def score_lstm(Xte, m, sc, bs=4096):
    m.eval()
    Xs  = sc.transform(Xte).astype(np.float32)
    out = []
    with torch.no_grad():
        for i in range(0, len(Xs), bs):
            b = torch.from_numpy(Xs[i:i+bs]).unsqueeze(1)
            o = m(b)
            out.append(torch.mean((b-o)**2, dim=(1,2)).numpy())
    return np.concatenate(out)

LSTMAE_results = []
for i, (fold, inj) in enumerate(zip(folds, injected_folds)):
    print(f'LSTM-AE | Fold {i+1}')
    d = fold['X_train'].shape[1]
    m, sc = train_lstm(fold['X_train'], d)
    s = score_lstm(inj['X_test_injected'], m, sc)
    print(f'  score min={s.min():.4f} max={s.max():.4f}')
    LSTMAE_results.append({'scores': s, 'y_true': inj['y_true']})
    pd.DataFrame({'anomaly_score': s, 'y_true': inj['y_true']}).to_csv(
        f'{OUTPUT_DIR}Report_Fold{i+1}_LSTM-AE_Scores_and_Classes.csv', index=False)
    print('  saved.')
print('LSTM-AE done.')


## Step 7 — Thresholding (μ+2σ) & Evaluation

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

model_names = ['DAGMM', 'IsolationForest', 'LOF', 'LSTM-AE']
all_results = [DAGMM_results, IF_results, LOF_results, LSTMAE_results]
eval_records = []

for mname, results in zip(model_names, all_results):
    print('\n===', mname, '===')
    for i, res in enumerate(results):
        s  = res['scores']; y = res['y_true']
        thr = s.mean() + 2*s.std()
        yp  = (s >= thr).astype(int)
        P   = precision_score(y, yp, zero_division=0)
        R   = recall_score(y, yp, zero_division=0)
        F1  = f1_score(y, yp, zero_division=0)
        print(f'  Fold {i+1}: thr={thr:.4f} | P={P:.4f} | R={R:.4f} | F1={F1:.4f} | pred_anom={yp.sum()}')
        eval_records.append({'Model': mname, 'Fold': i+1, 'Threshold': round(thr,6),
                             'Precision': round(P,4), 'Recall': round(R,4), 'F1': round(F1,4),
                             'Predicted_Anomalies': int(yp.sum()), 'Total_Test': len(y)})

df_eval = pd.DataFrame(eval_records)
df_eval.to_csv(f'{OUTPUT_DIR}Report_All_Folds_Evaluation_Metrics.csv', index=False)
print('\nSaved: Report_All_Folds_Evaluation_Metrics.csv')
print(df_eval.to_string(index=False))


## Step 8 — Bayesian Optimization (Subsampled 10K)

In [ ]:
from bayes_opt import BayesianOptimization
from sklearn.metrics import f1_score

# Subsample fold 0 to 10K rows for fast BO iterations
N_BO   = 10000
rng_bo = np.random.RandomState(42)
tr_idx = rng_bo.choice(len(folds[0]['X_train']), N_BO, replace=False)
te_idx = rng_bo.choice(len(injected_folds[0]['X_test_injected']),
                        min(N_BO, len(injected_folds[0]['y_true'])), replace=False)
Xtr_bo = folds[0]['X_train'][tr_idx]
Xte_bo = injected_folds[0]['X_test_injected'][te_idx]
yte_bo = injected_folds[0]['y_true'][te_idx]
opt_records = []
print(f'BO data: train={Xtr_bo.shape}, test={Xte_bo.shape}, anomalies={yte_bo.sum()}')

# ── IF BO ─────────────────────────────────────────────────────────────────
print('\n=== Bayesian Optimization: Isolation Forest ===')
def if_obj(n_estimators, max_samples_frac, contamination):
    m = IsolationForest(n_estimators=int(n_estimators),
                        max_samples=round(max_samples_frac,3),
                        contamination=round(contamination,3),
                        n_jobs=-1, random_state=42)
    m.fit(Xtr_bo); s=-m.score_samples(Xte_bo)
    thr=s.mean()+2*s.std(); yp=(s>=thr).astype(int)
    sc=f1_score(yte_bo,yp,zero_division=0)
    opt_records.append({'Model':'IsolationForest','n_estimators':int(n_estimators),
                        'max_samples_frac':round(max_samples_frac,3),
                        'contamination':round(contamination,3),'F1':sc})
    return sc
if_bo = BayesianOptimization(f=if_obj,
    pbounds={'n_estimators':(50,300),'max_samples_frac':(0.5,1.0),'contamination':(0.05,0.2)},
    random_state=42, verbose=2)
if_bo.maximize(init_points=5, n_iter=10)
print('Best IF:', if_bo.max)

# ── LOF BO ────────────────────────────────────────────────────────────────
print('\n=== Bayesian Optimization: LOF ===')
def lof_obj(n_neighbors, leaf_size, contamination):
    m = LocalOutlierFactor(n_neighbors=int(n_neighbors), leaf_size=int(leaf_size),
                           contamination=round(contamination,3), novelty=True, n_jobs=-1)
    m.fit(Xtr_bo); s=-m.score_samples(Xte_bo)
    thr=s.mean()+2*s.std(); yp=(s>=thr).astype(int)
    sc=f1_score(yte_bo,yp,zero_division=0)
    opt_records.append({'Model':'LOF','n_neighbors':int(n_neighbors),
                        'leaf_size':int(leaf_size),'contamination':round(contamination,3),'F1':sc})
    return sc
lof_bo = BayesianOptimization(f=lof_obj,
    pbounds={'n_neighbors':(5,50),'leaf_size':(10,60),'contamination':(0.05,0.2)},
    random_state=42, verbose=2)
lof_bo.maximize(init_points=5, n_iter=10)
print('Best LOF:', lof_bo.max)

df_opt = pd.DataFrame(opt_records)
df_opt.to_csv(f'{OUTPUT_DIR}Report_DBPO_Hyperparameter_Optimization.csv', index=False)
print('\nSaved: Report_DBPO_Hyperparameter_Optimization.csv')
print(df_opt.to_string(index=False))


## Step 9 — SHAP Analysis

In [ ]:
import shap
import matplotlib.pyplot as plt

avg_f1 = df_eval.groupby('Model')['F1'].mean().sort_values(ascending=False)
print('Average F1 per model:')
print(avg_f1.to_string())
print('Best model for SHAP:', avg_f1.idxmax())

Xtr_shap = folds[0]['X_train']
Xte_shap = injected_folds[0]['X_test_injected']
yte_shap = injected_folds[0]['y_true']

# Sample 3000 points for fast SHAP
n_s   = min(3000, len(Xte_shap))
Xte_s = Xte_shap[:n_s]
yte_s = yte_shap[:n_s]

shap_m = IsolationForest(n_estimators=100, contamination=0.1, n_jobs=-1, random_state=42)
shap_m.fit(Xtr_shap)
explainer = shap.TreeExplainer(shap_m)
sv = explainer.shap_values(Xte_s)
sv = sv[0] if isinstance(sv, list) else sv
print('SHAP values shape:', sv.shape)

mean_abs = np.abs(sv).mean(0)
top_idx  = int(np.argmax(mean_abs))
top_feat = feature_cols[top_idx]
print('Top feature:', top_feat)
print('Top 10:')
for j in np.argsort(mean_abs)[::-1][:10]:
    print(f'  {feature_cols[j]}: {mean_abs[j]:.4f}')

shap.dependence_plot(top_idx, sv, Xte_s, feature_names=feature_cols,
                     interaction_index=None, show=False)
plt.title('SHAP Dependency Plot - ' + top_feat)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}Report_SHAP_Dependency_Plot_Top_Feature.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: SHAP plot')

real_sv = sv[yte_s == 0]
anom_sv = sv[yte_s == 1]
df_contrast = pd.DataFrame({
    'Feature': feature_cols,
    'SHAP_Real_Mean':    np.mean(real_sv, axis=0),
    'SHAP_Anomaly_Mean': np.mean(anom_sv, axis=0),
    'SHAP_Difference':   np.mean(anom_sv,axis=0)-np.mean(real_sv,axis=0)
}).sort_values('SHAP_Difference', key=abs, ascending=False)
df_contrast.to_csv(f'{OUTPUT_DIR}Report_SHAP_Contrastive_Check.csv', index=False)
print('Saved: Report_SHAP_Contrastive_Check.csv')
print(df_contrast.to_string(index=False))


## Step 10 — Final Summary & Download All Reports

In [ ]:
import os, zipfile

files = sorted(os.listdir(OUTPUT_DIR))
print(f'Output: {OUTPUT_DIR}   Total files: {len(files)}')
for f in files:
    sz = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f}  ({sz/1024:.1f} KB)')

zip_path = '/content/BMM_SCI_46_Reports.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in files:
        zf.write(os.path.join(OUTPUT_DIR, f), f)
print(f'\nZipped: {zip_path}')

from google.colab import files
files.download(zip_path)
print('Download started!')
